# Can a quantum computer help clean galamsey rivers? 🇬🇭
**Qiskit Fall Fest hackathon demo — mercury, sulfur and illegal gold mining**

Illegal small-scale gold mining (*galamsey*) in Ghana uses mercury, which ends up in rivers. One way to pull mercury out of water is with filter materials rich in **sulfur**. But does mercury really prefer sulfur over oxygen (which is everywhere in water)?

We ask a tiny chemistry question with a (simulated) quantum computer:

**Hg(OH)₂ + 2 H₂S → Hg(SH)₂ + 2 H₂O**

If the energy change is **negative**, mercury would rather hold on to sulfur than oxygen — good news for sulfur-based filters.

> ⚠️ **Honesty note:** this is a *tiny teaching model* (8 qubits, a small slice of each molecule) that shows the full pipeline from molecule to quantum circuit to answer. It runs on a normal laptop simulator and is **not** a demonstration of quantum advantage. Real-world filter design would need far bigger calculations and lab tests.

---
**Step 0 — install the tools.** This first cell installs the free software we need (takes about a minute on Colab; skip if already installed).

In [ ]:
%pip install -q qiskit qiskit-aer qiskit-nature qiskit-algorithms pyscf matplotlib numpy scipy

**Step 1 — load the tools** and start a stopwatch so we can see how long the whole demo takes.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # small matrices run faster on one thread
import time, numpy as np, warnings
warnings.filterwarnings("ignore")
from pyscf import gto, scf, mcscf, ao2mo
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize
import matplotlib.pyplot as plt
HA2KCAL = 627.509   # converts the computer's energy unit into kcal/mol, the chemist's everyday unit
T_START = time.time()

**Step 2 — the four molecules.** These are the atom positions (in ångström, i.e. tenths of a billionth of a metre) for water, hydrogen sulfide, and the two mercury compounds. They were found earlier by letting a standard chemistry program relax each molecule into its most comfortable shape; we reuse them here to save time.

In [ ]:
GEOMETRIES = {
 "H2O": "O 0.0000 -0.0098 0.0000; H 0.7570 0.5913 0.0000; H -0.7570 0.5913 0.0000",
 "H2S": "S -0.0000 -0.0054 -0.0000; H 0.9739 0.9328 -0.0000; H -0.9739 0.9328 -0.0000",
 "Hg(OH)2": "Hg 0.0000 0.0000 -0.0000; O 1.9816 -0.0241 -0.0000; H 2.3138 0.8861 -0.0000; O -1.9816 0.0241 -0.0000; H -2.3138 -0.8861 -0.0000",
 "Hg(SH)2": "Hg -0.0000 0.0000 0.0000; S 2.3815 -0.0735 0.0000; H 2.5158 1.2750 -0.0000; S -2.3815 0.0735 0.0000; H -2.5158 -1.2750 -0.0000"
}

**Step 3 — trusted reference answers (precomputed).** Two well-established classical methods (B3LYP and CCSD(T), the latter often called the chemist's "gold standard") were run once beforehand on full molecules. They take a few minutes, so we just paste the results in. Energies are in *hartree*, the natural unit of chemistry.

In [ ]:
PRECOMPUTED = {
 "H2O": {
  "B3LYP": -76.35831575416277,
  "CCSD(T)": -76.17703496921959
 },
 "H2S": {
  "B3LYP": -399.28224994393275,
  "CCSD(T)": -398.7441721381719
 },
 "Hg(OH)2": {
  "B3LYP": -305.034014329451,
  "CCSD(T)": -303.9982444430431
 },
 "Hg(SH)2": {
  "B3LYP": -950.9441179053372,
  "CCSD(T)": -949.2027911973366
 }
}
print('Precomputed reference energies loaded (labelled as precomputed, not run live).')

**Step 4 — the recipe for one molecule.** For each molecule we:
1. Do a quick classical calculation of where the electrons sit (called Hartree–Fock).
2. Zoom in on the 4 most important electrons in 4 "seats" (orbitals) — the part where bonding happens. That small slice fits on **8 qubits**.
3. Get the **exact** answer for that slice the classical way (CASCI), as a referee.
4. Run the **quantum algorithm (VQE)**: a circuit with adjustable knobs, and a classical optimiser that turns the knobs until the energy is as low as possible.
5. Double-check the final knob settings on a genuine Qiskit circuit.

In [ ]:
def run_molecule(name, geometry):
    t0 = time.time()
    mol = gto.M(atom=geometry, basis="def2-svp", ecp={"Hg": "def2-svp"}, verbose=0)
    hf = scf.RHF(mol).run()                                   # 1. quick classical starting point
    cas = mcscf.CASCI(hf, 4, 4); cas.verbose = 0             # 2. 4 electrons in 4 orbitals
    e_casci = cas.kernel()[0]                                 # 3. exact answer for the slice
    h1, ecore = cas.get_h1eff(); h2 = ao2mo.restore(1, cas.get_h2eff(), 4)
    mapper = JordanWignerMapper()
    H = mapper.map(ElectronicEnergy.from_raw_integrals(h1, h2).second_q_op())   # energy as qubit operator
    ansatz = UCCSD(4, (2, 2), mapper, initial_state=HartreeFock(4, (2, 2), mapper))
    # 4. VQE: fast exact simulation of the same circuit, optimised classically
    Hm = H.to_matrix(); eig = [np.linalg.eigh(G.to_matrix()) for G in ansatz.operators]
    psi0 = Statevector(ansatz.initial_state).data
    def energy(theta):
        v = psi0.copy()
        for th, (lam, V) in zip(theta, eig):
            v = V @ (np.exp(-1j * th * lam) * (V.conj().T @ v))
        return float(np.real(np.vdot(v, Hm @ v)))
    opt = minimize(energy, np.zeros(ansatz.num_parameters), method="L-BFGS-B",
                   options={"ftol": 1e-12, "gtol": 1e-6, "maxiter": 500})
    # 5. check on the real Qiskit circuit
    e_circ = float(StatevectorEstimator().run([(ansatz, H, opt.x)]).result()[0].data.evs)
    e_vqe = e_circ + ecore
    print(f"{name:8s} qubits={H.num_qubits} knobs={ansatz.num_parameters} | "
          f"VQE={e_vqe:.6f}  exact={e_casci:.6f}  gap={(e_vqe-e_casci)*1000:.5f} mHa  ({time.time()-t0:.1f}s)")
    return {"HF": hf.e_tot, "CASCI": e_casci, "VQE": e_vqe}

**Step 5 — run it live for all four molecules.** Watch the VQE energy match the exact referee answer (the "gap" should be tiny).

In [ ]:
results = {}
for name, g in GEOMETRIES.items():
    results[name] = {**run_molecule(name, g), **PRECOMPUTED[name]}

**Step 6 — the reaction energy.** We add up the products and subtract the starting materials:

energy change = [Hg(SH)₂ + 2 H₂O] − [Hg(OH)₂ + 2 H₂S]

A **negative** number means mercury prefers sulfur.

In [ ]:
def reaction(m):
    r = results
    return (r["Hg(SH)2"][m] + 2*r["H2O"][m] - r["Hg(OH)2"][m] - 2*r["H2S"][m]) * HA2KCAL
methods = ["HF", "CASCI", "VQE", "B3LYP", "CCSD(T)"]
labels = {"HF": "Hartree-Fock (live)", "CASCI": "Exact, 8-qubit slice (live)", "VQE": "Quantum VQE, 8 qubits (live)",
          "B3LYP": "B3LYP (precomputed)", "CCSD(T)": "CCSD(T) gold standard (precomputed)"}
dE = {m: reaction(m) for m in methods}
for m in methods:
    print(f"{labels[m]:38s} {dE[m]:8.2f} kcal/mol")
print(f"\nVQE vs exact slice: {abs(dE['VQE']-dE['CASCI']):.5f} kcal/mol apart")

**Step 7 — the picture.** Every method gives a clearly negative bar: all agree that mercury prefers sulfur.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#9e9e9e", "#1f77b4", "#7b2cbf", "#ff9f1c", "#2a9d8f"]
bars = ax.bar([labels[m].replace(" (", "\n(") for m in methods], [dE[m] for m in methods], color=colors)
for b, m in zip(bars, methods):
    ax.text(b.get_x()+b.get_width()/2, dE[m]-1, f"{dE[m]:.1f}", ha="center", va="top", color="white", fontweight="bold")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Energy change (kcal/mol)\nnegative = mercury prefers sulfur")
ax.set_title("Hg(OH)2 + 2 H2S -> Hg(SH)2 + 2 H2O")
plt.xticks(fontsize=8); plt.tight_layout(); plt.show()
print(f"Total demo run time: {time.time()-T_START:.1f} s")

## What did we learn?
- The quantum algorithm (VQE) reproduces the exact answer for its 8-qubit slice almost perfectly.
- Every method — quantum and classical — says swapping oxygen for sulfur around mercury **releases energy** (roughly −40 to −50 kcal/mol). Mercury strongly prefers sulfur.
- That supports using **sulfur-rich (thiol) filter materials** to capture mercury from galamsey-polluted rivers.

**Limits, plainly:** this is a toy-sized model run on a simulator. A normal laptop can solve it exactly, so there is **no quantum advantage here**. The value is the pipeline: the same steps could scale to larger, harder molecules as quantum hardware improves.